In [1]:
sample = 'mouse_skin'
save_dir = 'raman_vis'

In [ ]:
import cv2
import glob
import numpy as np
import pandas as pd
import scipy.io
from scipy.ndimage import rotate
from skimage import transform as tf
from skimage.transform import warp
import scipy.io
import cv2
import matplotlib.pyplot as plt
import os

wave_number = scipy.io.loadmat(f'{sample}/wavenumbers.mat')['wavenumber'][0][413:1286].round(0).astype(np.int16)

In [ ]:
from scipy.integrate import simpson
from scipy.stats import ranksums
from statsmodels.stats.multitest import multipletests

def normalize_spectra(wavenum, intensity, min_peak, max_peak):
    amide_mask = (wavenum >= min_peak) & (wavenum <= max_peak)
    area = simpson(intensity[amide_mask], x=wavenum[amide_mask])
    # print(area)
    return intensity / area

In [5]:
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon
import tqdm
from scipy.spatial import ConvexHull
import pandas as pd


def generate_peak_map(sample, save_dir, selected_peak, selected_marker, selected_cell_type, selected_linewidth):
    
    all_sample_path = [ 
                        f'./{sample}/Sample 1/STARmap/',
                        f'./{sample}/Sample 2/STARmap/',
                        f'./{sample}/Sample 3/STARmap/',
                        f'./{sample}/Sample 4/STARmap/',
                        f'./{sample}/Sample 5/STARmap/',
                        f'./{sample}/Sample 6/STARmap/',
                        ]

    for sample_path in all_sample_path:

        sample_name = sample_path.split('/')[2].split(' ')[1] # 1 to 6
        # get raman features
        raman_features = pd.read_csv(sample_path + 'cell_raman.csv', index_col=0) # file 1 
        # normalize raman features
        new_features = []
        raman_features_values = raman_features.values
        for i in range(raman_features_values.shape[0]):
            new_features.append(normalize_spectra(wave_number, raman_features_values[i], 1630, 1700))
        raman_features_values = np.stack(new_features, axis=0)
        raman_features.iloc[:, :] = raman_features_values 
        # get selected peak
        raman_features = raman_features[selected_peak]

        # get selected N cells
        cell_mapping = pd.read_csv(sample_path + 'cell_mapping.csv', index_col=0) # file 2
        if selected_cell_type != 'global':
            cell_mapping = cell_mapping[cell_mapping['cell_type'] == selected_cell_type]
        # selected_sen_cells = cell_mapping[cell_mapping[selected_marker]]
    
        # get only senescent cells
        # cell_mapping = cell_mapping[cell_mapping['p21+'] == True]

        # get starmap dapi image
        starmap_dapi_name = glob.glob(sample_path + '*_v3.tif') # file 3
        starmap_image = cv2.imread(starmap_dapi_name[0], -1)


        if sample == 'mouse_lung':
            tranform_matrix_name = glob.glob(sample_path + 'matrix.mat') # file 5
        else:
            tranform_matrix_name = glob.glob(sample_path + 'matrix_*.mat') # file 5

        t = scipy.io.loadmat(tranform_matrix_name[0])['matrix']
        tform = tf.AffineTransform(matrix=t)
        raman_image_name = glob.glob(sample_path + '*_T333.mat') # file 6
        selected_index = np.where(wave_number == int(selected_peak))[0][0]
        raman_image = scipy.io.loadmat(raman_image_name[0])['T333'][:, :, selected_index]
        
        if sample == 'mouse_skin':
            if 'Sample 1' in sample_path :
                raman_image = rotate(raman_image, 90, reshape=False)
            else:
                raman_image = rotate(raman_image, 180, reshape=False)
        else:
            raman_image = rotate(raman_image, 90, reshape=False)

        aligned_raman_image = warp(raman_image, tform, output_shape=starmap_image.shape)
       
        # Load segmentation mask
        starmap_segmentation_name = glob.glob(sample_path + '*segmentation.tif')  # file 4
        starmap_mask = cv2.imread(starmap_segmentation_name[0], -1)
        # Get the allowed cell IDs
        keep_cell_dix = np.array([int(cell.split('_')[1]) for cell in cell_mapping.index])
        # Create a boolean mask of valid cells
        valid_mask = np.isin(starmap_mask, keep_cell_dix)
        # Zero out everything else
        starmap_mask[~valid_mask] = 0


        aligned_raman_image = aligned_raman_image * (starmap_mask>0)
        # normalize raman image
        aligned_raman_image = (aligned_raman_image - aligned_raman_image.min()) / (aligned_raman_image.max() - aligned_raman_image.min())

        # plot
        plt.imshow(aligned_raman_image, cmap='jet', vmin = np.percentile(aligned_raman_image, 1), vmax = np.percentile(aligned_raman_image, 99))
        # colorbar
        plt.colorbar()

        selected_cells = cell_mapping[cell_mapping['p21+'] == True]

        # Overlay the white boundaries by re-plotting them
        for cell in tqdm.tqdm(selected_cells.index):
            mask = starmap_mask == int(cell.split('_')[1])
            y, x = np.where(mask)
            hull = ConvexHull(np.array([x, y]).T)
            hull_points = np.array([x, y]).T[hull.vertices]
            # add the first point to the end to close the polygon
            hull_points = np.vstack([hull_points, hull_points[0]])
            plt.plot(hull_points[:, 0], hull_points[:, 1], color='white', linewidth=selected_linewidth)

        # Save the image without axis
        plt.axis('off')
        plt.tight_layout()

        plt.savefig(f'figures/{sample}/{save_dir}/sample_' + str(sample_name) + '_' + selected_marker + '_' + selected_cell_type + '_peak_' + selected_peak + '_width_' + str(selected_linewidth) + '_raman.pdf', bbox_inches='tight', pad_inches=0, dpi=4000)
        plt.show()

    # break


In [ ]:
from joblib import Parallel, delayed

sample = 'mouse_skin'
save_dir = 'raman_vis'
length = 0.1

if not os.path.exists(f'figures/{sample}/{save_dir}'):
    os.makedirs(f'figures/{sample}/{save_dir}')

# Define peaks for each label
global_peaks = [1134, 1135, 1132, 1137, 1131, 1130, 1162, 1164, 937, 940, 942, 934]
ife_peaks = [1134, 1135, 1132, 1137, 1131, 1130, 940, 937, 942, 936, 943, 939]

# Filter only those peaks present in wave_number
global_peaks = [p for p in global_peaks if p in wave_number]
ife_peaks = [p for p in ife_peaks if p in wave_number]

# Combine into task list
tasks = [('global', peak) for peak in global_peaks] + \
        [('Interfollicular Epidermis cell', peak) for peak in ife_peaks]

# Define task function
def run_task(label, peak):
    generate_peak_map( sample, save_dir, str(peak), 'all', label, length)

# Execute in parallel
Parallel(n_jobs=24)(delayed(run_task)(label, peak) for label, peak in tasks)

